# Codebook Inspection — JEPA-Coder Phase 2 Checkpoint

**Purpose**: Verify the go/no-go decision after SST training (arch v2 §4.2).

For each VQ codebook entry, this notebook finds the 20 training blocks whose Reasoner
output vector was closest to that entry. If entries correspond to recognizable code
patterns (input parsing, loops, DP, output formatting), proceed to Talker training.
If entries are random, debug before continuing.

**Inputs** (configure in the cell below):
- `checkpoints/sst/sst_reasoner_final.pt`
- `checkpoints/sst/vq_codebook_final.pt`
- `data/sst_dataset.jsonl` (1000 examples sampled)

**Arch reference**: `docs/jepa_coder_architecture_v2.md` §4.2, §9 Tests 6–8

In [6]:
import sys
import os

# 1. Environment Setup
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Setting up workspace...")
    
    # Clone the repo into the Colab environment so it can see 'models/'
    if not os.path.exists('/content/jepa-coder'):
        !git clone https://github.com/pu-suo/jepa-coder.git /content/jepa-coder
    else:
        # Pull latest changes if it already exists
        !cd /content/jepa-coder && git pull

    # Set REPO_ROOT to the cloned directory
    REPO_ROOT = '/content/jepa-coder'
    
except ImportError:
    IN_COLAB = False
    print("Running Locally.")
    from pathlib import Path
    REPO_ROOT = Path.cwd()
    if REPO_ROOT.name == "notebooks":
        REPO_ROOT = REPO_ROOT.parent
    REPO_ROOT = str(REPO_ROOT)

# Add repo to Python path
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Repo root configured as: {REPO_ROOT}")

# ── Paths (You will need to update these for Colab!) ─────────────────────────
# If your checkpoints and data are large, you likely need to mount Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Example paths assuming you uploaded them to a 'jepa_coder_data' folder in Drive:
    REASONER_CKPT = "/content/drive/MyDrive/jepa_coder_data/checkpoints/sst/sst_reasoner_final.pt"
    VQ_CKPT       = "/content/drive/MyDrive/jepa_coder_data/checkpoints/sst/vq_codebook_final.pt"
    DATA_JSONL    = "/content/drive/MyDrive/jepa_coder_data/data/sst_dataset.jsonl"
else:
    REASONER_CKPT = os.path.join(REPO_ROOT, "checkpoints/sst/sst_reasoner_final.pt")
    VQ_CKPT       = os.path.join(REPO_ROOT, "checkpoints/sst/vq_codebook_final.pt")
    DATA_JSONL    = os.path.join(REPO_ROOT, "data/sst_dataset.jsonl")

N_EXAMPLES    = 1000
TOP_K         = 20
DISPLAY_ENTRIES = 20

print(f"Reasoner checkpoint : {REASONER_CKPT}")
print(f"VQ checkpoint       : {VQ_CKPT}")
print(f"Data                : {DATA_JSONL}")

Mounted at /content/drive
Reasoner checkpoint : /content/drive/MyDrive/jepa_coder_data/checkpoints/sst/sst_reasoner_final.pt
VQ checkpoint       : /content/drive/MyDrive/jepa_coder_data/checkpoints/sst/vq_codebook_final.pt
Data                : /content/drive/MyDrive/jepa_coder_data/data/sst_dataset.jsonl


## 1. Load Models

In [7]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

from models.reasoner import Reasoner
from models.vq import VectorQuantizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b", trust_remote_code=True)
VOCAB_SIZE = len(tokenizer)
print(f"Vocab size: {VOCAB_SIZE}")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 49152


In [8]:
# ── Reasoner ──────────────────────────────────────────────────────────────────
reasoner = Reasoner(
    vocab_size=VOCAB_SIZE,
    dim=768,
    n_layers=16,
    n_heads=12,
    ffn_dim=3072,
    max_seq_len=1024,
)

raw = torch.load(REASONER_CKPT, map_location="cpu")
state_dict = raw.get("model_state_dict", raw) if isinstance(raw, dict) else raw
missing, unexpected = reasoner.load_state_dict(state_dict, strict=False)
non_lmhead = [k for k in unexpected if not k.startswith("lm_head")]
assert not non_lmhead, f"Unexpected non-lm_head keys: {non_lmhead}"
assert not missing,    f"Missing keys: {missing}"

# SST mode: L2 norm enabled, no LM head
reasoner.hybrid_norm.l2_enabled = True
reasoner.lm_head = None
reasoner.eval()
reasoner.requires_grad_(False)
reasoner = reasoner.to(device)

print(f"Reasoner loaded from {REASONER_CKPT.name}")
n_params = sum(p.numel() for p in reasoner.parameters())
print(f"Parameters: {n_params / 1e6:.1f}M")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/jepa_coder_data/checkpoints/sst/sst_reasoner_final.pt'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── VQ ────────────────────────────────────────────────────────────────────────
vq = VectorQuantizer(codebook_size=512, dim=768, commitment_cost=0.25)

raw_vq = torch.load(VQ_CKPT, map_location="cpu")
state_dict_vq = raw_vq.get("model_state_dict", raw_vq) if isinstance(raw_vq, dict) else raw_vq
vq.load_state_dict(state_dict_vq)
vq.eval()
vq.requires_grad_(False)
vq = vq.to(device)

CODEBOOK_SIZE = vq.codebook_size
print(f"VQ loaded from {VQ_CKPT.name}")
print(f"Codebook size: {CODEBOOK_SIZE}")

## 2. Run Frozen Reasoner on 1000 Examples

For each example we collect, per code block:
- the Reasoner output vector `r` (768-dim, unit norm)
- the VQ codebook index assigned
- the block's source code
- the block's AST type
- the parent problem text (first 200 chars)

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import List
import textwrap

@dataclass
class BlockRecord:
    vq_idx:       int
    cos_sim:      float   # dot(r, codebook[vq_idx])  — since both unit norm
    r_vector:     torch.Tensor   # (768,) on CPU
    block_code:   str
    block_type:   str
    problem_text: str    # truncated
    example_idx:  int
    block_pos:    int    # position in this example's block sequence


all_records: List[BlockRecord] = []
# per_entry[k] → list of BlockRecord (all records with vq_idx == k)
per_entry: defaultdict[int, List[BlockRecord]] = defaultdict(list)

skipped = 0
processed = 0
total_blocks = 0

In [ ]:
from tqdm.auto import tqdm

with open(DATA_JSONL, encoding="utf-8") as f:
    lines = [f.readline() for _ in range(N_EXAMPLES * 2)]   # over-read in case of skips

with torch.inference_mode():
    pbar = tqdm(total=N_EXAMPLES, desc="Running Reasoner")

    for raw_line in lines:
        if processed >= N_EXAMPLES:
            break
        if not raw_line.strip():
            continue

        row = json.loads(raw_line)
        blocks = json.loads(row["blocks_json"])
        problem_text = row["problem"]

        # Filter: only examples with at least one CODE block
        code_blocks = [b for b in blocks if b["type"] == "CODE"]
        if not code_blocks:
            skipped += 1
            continue

        # Tokenize problem (truncate to 1024)
        prob_ids = tokenizer(
            problem_text,
            return_tensors="pt",
            add_special_tokens=True,
            truncation=True,
            max_length=1024,
        ).input_ids.squeeze(0).to(device)

        # Encode problem → h_0
        h = reasoner.encode_problem(prob_ids)   # (768,)

        for block_pos, block in enumerate(blocks):
            # Reasoner step
            r = reasoner.step(h)   # (768,), unit norm

            # VQ
            _, idx, _ = vq(r)
            idx_int = idx.item()
            codebook_vec = vq.embedding.weight[idx_int]   # (768,)
            cos_sim = torch.dot(r, codebook_vec).item()   # both unit norm

            rec = BlockRecord(
                vq_idx=idx_int,
                cos_sim=cos_sim,
                r_vector=r.cpu().clone(),
                block_code=block.get("code", ""),
                block_type=block.get("type", "CODE"),
                problem_text=problem_text[:200],
                example_idx=processed,
                block_pos=block_pos,
            )
            all_records.append(rec)
            per_entry[idx_int].append(rec)
            total_blocks += 1

            # Loop back continuous state (§3.5: h = r, NOT quantized)
            h = r

        processed += 1
        pbar.update(1)

    pbar.close()

print(f"\nProcessed : {processed} examples")
print(f"Skipped   : {skipped} examples (no code blocks)")
print(f"Blocks    : {total_blocks} total")
print(f"Unique VQ entries used: {len(per_entry)} / {CODEBOOK_SIZE}")

## 3. Utilization Statistics

Architecture spec §9 Test 6: >30% of entries used across 1000 examples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

usage = np.zeros(CODEBOOK_SIZE, dtype=int)
for rec in all_records:
    usage[rec.vq_idx] += 1

n_used = (usage > 0).sum()
utilization = n_used / CODEBOOK_SIZE
print(f"Entries used      : {n_used} / {CODEBOOK_SIZE}  ({utilization*100:.1f}%)")
print(f"Test 6 threshold  : 30%  →  {'PASS ✓' if utilization >= 0.30 else 'FAIL ✗'}")
print(f"")
print(f"Most-used entry   : {usage.argmax()}  ({usage.max()} blocks)")
print(f"Mean blocks/entry : {usage[usage > 0].mean():.1f}  (used entries only)")
print(f"Median            : {np.median(usage[usage > 0]):.1f}")

# Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.bar(range(CODEBOOK_SIZE), np.sort(usage)[::-1], width=1.0, color="steelblue", alpha=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Codebook entry (sorted by usage)")
ax.set_ylabel("Block count")
ax.set_title("Usage distribution (sorted)")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

ax2 = axes[1]
nonzero = usage[usage > 0]
ax2.hist(nonzero, bins=40, color="steelblue", alpha=0.8, edgecolor="white")
ax2.set_xlabel("Blocks assigned to entry")
ax2.set_ylabel("Number of entries")
ax2.set_title(f"Usage histogram (used entries only, n={len(nonzero)})")

plt.suptitle("VQ Codebook Utilization", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Top-20 Blocks per Codebook Entry

For each codebook entry, we rank all blocks that were assigned to it by cosine similarity
between the Reasoner output `r` and the codebook vector. The closest blocks are the most
prototypical examples of what that entry represents.

Architecture spec §9 Test 7: entries should correspond to recognizable code patterns.

In [ ]:
# For each codebook entry, also compute the top-K from ALL records by direct
# cosine similarity to the codebook vector — this finds the closest blocks
# even if they were ultimately assigned to a neighbouring entry.
# We also provide the simpler 'assigned' view: blocks that were actually quantized
# to this entry, sorted by cos_sim descending.

codebook_vecs = vq.embedding.weight.detach().cpu()   # (512, 768)
r_stack = torch.stack([rec.r_vector for rec in all_records])  # (N_blocks, 768)

def top_k_for_entry(entry_idx: int, k: int = TOP_K, mode: str = "assigned") -> List[BlockRecord]:
    """
    mode='assigned': only blocks assigned to this entry, sorted by cos_sim descending
    mode='global':   all blocks, ranked by dot(r, codebook[entry_idx]) descending
    """
    if mode == "assigned":
        candidates = per_entry[entry_idx]
        return sorted(candidates, key=lambda rec: -rec.cos_sim)[:k]
    else:
        c = codebook_vecs[entry_idx]                          # (768,)
        sims = (r_stack @ c).numpy()                          # (N_blocks,)
        top_indices = np.argsort(sims)[::-1][:k]
        return [all_records[i] for i in top_indices]

In [ ]:
# Sort entries by usage (most used first) for display
entries_by_usage = sorted(per_entry.keys(), key=lambda k: -usage[k])

def display_entry(entry_idx: int, n_show: int = TOP_K) -> None:
    records = top_k_for_entry(entry_idx, k=n_show, mode="assigned")
    print("═" * 80)
    print(f"  ENTRY {entry_idx:>3d}  │  {usage[entry_idx]} blocks assigned across {N_EXAMPLES} examples")
    print("═" * 80)
    for rank, rec in enumerate(records, 1):
        print(f"  [{rank:>2d}] cos_sim={rec.cos_sim:.4f}  example={rec.example_idx}  "
              f"block_pos={rec.block_pos}  type={rec.block_type}")
        print(f"       problem: {rec.problem_text[:100].replace(chr(10), ' ')}")
        code_lines = rec.block_code.strip().split("\n")
        if len(code_lines) <= 6:
            for line in code_lines:
                print(f"         {line}")
        else:
            for line in code_lines[:5]:
                print(f"         {line}")
            print(f"         ... ({len(code_lines) - 5} more lines)")
        print()

# Display the top DISPLAY_ENTRIES most-used entries
for entry_idx in entries_by_usage[:DISPLAY_ENTRIES]:
    display_entry(entry_idx, n_show=5)   # 5 examples inline; full list available below

In [ ]:
# ── Full top-20 for a specific entry ──────────────────────────────────────────
# Change INSPECT_ENTRY to drill into any entry.

INSPECT_ENTRY = entries_by_usage[0]   # default: most-used entry

print(f"Full top-{TOP_K} for entry {INSPECT_ENTRY}\n")
display_entry(INSPECT_ENTRY, n_show=TOP_K)

## 5. Block-Type Distribution per Entry

How does the AST block type (For, If, Assign, FunctionDef, …) distribute across entries?
If the codebook is meaningful, certain entries should be dominated by specific block types.

In [ ]:
from collections import Counter

# Infer block type from the code text as a proxy
# (the stored block_type is 'CODE' or 'STOP'; AST type isn't stored in sst_dataset.jsonl)
def _infer_type(code: str) -> str:
    s = code.strip()
    if s.startswith("def ") or s.startswith("async def "):
        return "FunctionDef"
    if s.startswith("class "):
        return "ClassDef"
    if s.startswith("for "):
        return "For"
    if s.startswith("while "):
        return "While"
    if s.startswith("if "):
        return "If"
    if s.startswith("try:"):
        return "Try"
    if s.startswith("with "):
        return "With"
    if s.startswith("return ") or s == "return":
        return "Return"
    if s.startswith("import ") or s.startswith("from "):
        return "Import"
    if s.startswith("<STOP>"):
        return "STOP"
    return "SimpleStmt"

# Build per-entry type counter
entry_type_counts: dict[int, Counter] = {}
for entry_idx, records in per_entry.items():
    counter = Counter(_infer_type(rec.block_code) for rec in records)
    entry_type_counts[entry_idx] = counter

# Show dominant type for top entries
print(f"{'Entry':>6}  {'Count':>6}  {'Dominant type':<14}  {'%':>5}  Distribution")
print("-" * 70)
for entry_idx in entries_by_usage[:30]:
    counter = entry_type_counts[entry_idx]
    total = sum(counter.values())
    top_type, top_cnt = counter.most_common(1)[0]
    pct = top_cnt / total * 100
    dist = "  ".join(f"{t}:{c}" for t, c in counter.most_common(4))
    print(f"  {entry_idx:>4d}  {total:>6d}  {top_type:<14}  {pct:>4.0f}%  {dist}")

## 6. Cosine Similarity Statistics

How tight are the clusters? A well-trained VQ should have high mean cosine similarity
between assigned vectors and their codebook entry.

In [ ]:
all_cos_sims = np.array([rec.cos_sim for rec in all_records])

print(f"Cosine similarity (r → assigned codebook entry)")
print(f"  Mean   : {all_cos_sims.mean():.4f}")
print(f"  Median : {np.median(all_cos_sims):.4f}")
print(f"  Std    : {all_cos_sims.std():.4f}")
print(f"  Min    : {all_cos_sims.min():.4f}")
print(f"  Max    : {all_cos_sims.max():.4f}")
print(f"  > 0.9  : {(all_cos_sims > 0.9).mean()*100:.1f}%  of blocks")
print(f"  > 0.7  : {(all_cos_sims > 0.7).mean()*100:.1f}%  of blocks")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_cos_sims, bins=60, color="steelblue", alpha=0.8, edgecolor="white")
ax.axvline(all_cos_sims.mean(), color="red", linestyle="--", label=f"mean={all_cos_sims.mean():.3f}")
ax.set_xlabel("Cosine similarity (r · codebook entry)")
ax.set_ylabel("Block count")
ax.set_title("Quantization tightness: r → assigned entry")
ax.legend()
plt.tight_layout()
plt.show()

## 7. t-SNE Visualization of the Codebook

Project the 512 codebook vectors (768-dim) to 2D. Each point is one codebook entry.
Color = log(usage count). Entries that are never used are shown in grey.

In [ ]:
from sklearn.manifold import TSNE
import warnings

print("Running t-SNE on codebook vectors (512 × 768)…")
codebook_np = codebook_vecs.numpy()   # (512, 768)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tsne = TSNE(
        n_components=2,
        perplexity=30,
        n_iter=1000,
        random_state=42,
        init="pca",
        learning_rate="auto",
        verbose=1,
    )
    coords = tsne.fit_transform(codebook_np)   # (512, 2)

print(f"t-SNE complete. KL divergence: {tsne.kl_divergence_:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: colored by log-usage ────────────────────────────────────────────────
ax = axes[0]

log_usage = np.log1p(usage)   # log(1 + count); 0 → 0 for unused entries
used_mask   = usage > 0
unused_mask = ~used_mask

# Unused entries in grey
if unused_mask.any():
    ax.scatter(
        coords[unused_mask, 0], coords[unused_mask, 1],
        c="lightgrey", s=18, zorder=1, label="Unused",
    )

# Used entries colored by log-usage
sc = ax.scatter(
    coords[used_mask, 0], coords[used_mask, 1],
    c=log_usage[used_mask], cmap="plasma", s=35,
    vmin=0, vmax=log_usage.max(), zorder=2,
)
fig.colorbar(sc, ax=ax, label="log(1 + blocks assigned)")

# Annotate top-5 most-used entries
for entry_idx in entries_by_usage[:5]:
    ax.annotate(
        str(entry_idx),
        xy=(coords[entry_idx, 0], coords[entry_idx, 1]),
        fontsize=7, fontweight="bold",
        xytext=(4, 4), textcoords="offset points",
    )

ax.set_title("Codebook t-SNE — colored by usage")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.legend(loc="lower right", fontsize=8)

# ── Right: colored by dominant AST block type ─────────────────────────────────
ax2 = axes[1]

TYPE_COLORS = {
    "For":         "#e41a1c",
    "If":          "#377eb8",
    "While":       "#ff7f00",
    "FunctionDef": "#4daf4a",
    "SimpleStmt":  "#984ea3",
    "Return":      "#a65628",
    "Import":      "#f781bf",
    "ClassDef":    "#999999",
    "Try":         "#66c2a5",
    "With":        "#fc8d62",
    "STOP":        "#8da0cb",
}
DEFAULT_COLOR = "#cccccc"

# Unused entries
if unused_mask.any():
    ax2.scatter(
        coords[unused_mask, 0], coords[unused_mask, 1],
        c="lightgrey", s=18, zorder=1, alpha=0.4,
    )

# Group used entries by dominant type and plot together
dom_type_per_entry = {}
for entry_idx in entries_by_usage:
    dom_type = entry_type_counts[entry_idx].most_common(1)[0][0]
    dom_type_per_entry[entry_idx] = dom_type

# Plot by type
for block_type, color in TYPE_COLORS.items():
    indices = [idx for idx in entries_by_usage if dom_type_per_entry.get(idx) == block_type]
    if not indices:
        continue
    ax2.scatter(
        coords[indices, 0], coords[indices, 1],
        c=color, s=35, label=block_type, zorder=2, alpha=0.85,
    )

ax2.set_title("Codebook t-SNE — colored by dominant block type")
ax2.set_xlabel("t-SNE dim 1")
ax2.set_ylabel("t-SNE dim 2")
ax2.legend(fontsize=8, loc="lower right", ncol=2)

plt.suptitle(
    f"VQ Codebook — {n_used}/{CODEBOOK_SIZE} entries used ({utilization*100:.1f}%) "
    f"over {N_EXAMPLES} examples",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

## 8. Inter-entry Similarity Heatmap (Top-50 Entries)

Pairwise cosine similarity between the most-used codebook entries. Entries that are
semantically related should cluster together in the heatmap.

In [ ]:
import seaborn as sns

N_HEAT = min(50, len(entries_by_usage))
top_indices = entries_by_usage[:N_HEAT]
top_vecs = codebook_vecs[top_indices]   # (N_HEAT, 768)

# Pairwise cosine similarity matrix (unit vectors → dot product)
sim_matrix = (top_vecs @ top_vecs.T).numpy()   # (N_HEAT, N_HEAT)

tick_labels = [
    f"{idx}\n({dom_type_per_entry.get(idx, '?')[:3]},{usage[idx]})"
    for idx in top_indices
]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    sim_matrix,
    xticklabels=tick_labels,
    yticklabels=tick_labels,
    cmap="RdYlBu_r",
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0.2,
    ax=ax,
    cbar_kws={"label": "Cosine similarity", "shrink": 0.8},
)
ax.set_title(f"Pairwise cosine similarity — top {N_HEAT} codebook entries (format: idx / dom-type / usage)")
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6, rotation=0)
plt.tight_layout()
plt.show()

## 9. STOP Detection Rate

Architecture spec §9 Test 8: >90% of examples should produce the STOP index (0)
as the last codebook entry.

In [ ]:
# Group records by example_idx, find the last block's vq_idx
from collections import defaultdict

last_block: dict[int, BlockRecord] = {}
for rec in all_records:
    if rec.example_idx not in last_block or rec.block_pos > last_block[rec.example_idx].block_pos:
        last_block[rec.example_idx] = rec

stop_idx = 0   # codebook entry 0 is reserved for STOP (arch v2 §2.2)
n_stop_correct = sum(1 for rec in last_block.values() if rec.vq_idx == stop_idx)
stop_rate = n_stop_correct / len(last_block)

print(f"STOP detection (last block → entry {stop_idx})")
print(f"  Correct     : {n_stop_correct} / {len(last_block)}")
print(f"  Rate        : {stop_rate*100:.1f}%")
print(f"  Threshold   : 90%  →  {'PASS ✓' if stop_rate >= 0.90 else 'FAIL ✗  (investigate before proceeding)'}")

# Distribution of what entry 'last blocks' get assigned to
last_idx_counts = Counter(rec.vq_idx for rec in last_block.values())
print(f"\n  Top-5 entries for last blocks:")
for idx, cnt in last_idx_counts.most_common(5):
    marker = " ← STOP" if idx == stop_idx else ""
    print(f"    entry {idx:>3d}: {cnt:>4d} examples{marker}")

## 10. Go / No-Go Summary

Arch spec §4.2: _"Inspect the codebook before proceeding. If entries correspond to
recognizable patterns… proceed. If entries are random, debug before continuing."_

In [ ]:
UTIL_THRESH   = 0.30
STOP_THRESH   = 0.90

checks = [
    ("Test 6: Utilization ≥ 30%",       utilization >= UTIL_THRESH,  f"{utilization*100:.1f}% used"),
    ("Test 8: STOP detection ≥ 90%",    stop_rate >= STOP_THRESH,    f"{stop_rate*100:.1f}%"),
    ("Mean cosine sim > 0.5",           all_cos_sims.mean() > 0.5,   f"{all_cos_sims.mean():.4f}"),
]

print("─" * 60)
print("  GO / NO-GO CHECKLIST")
print("─" * 60)
all_pass = True
for name, passed, value in checks:
    status = "PASS ✓" if passed else "FAIL ✗"
    if not passed:
        all_pass = False
    print(f"  {status}  {name:<35}  ({value})")
print("─" * 60)
print()
print("  Test 7 (codebook interpretability): MANUAL INSPECTION REQUIRED")
print("  → Review the block displays above.")
print("  → If top-20 blocks per entry share recognizable structure")
print("    (e.g. all For-loops, all input parsers), mark as PASS.")
print()
if all_pass:
    print("  Automatic checks: ALL PASS — proceed with manual Test 7.")
else:
    print("  Automatic checks: FAILURES DETECTED — debug before proceeding.")
print("─" * 60)